# Lots of match-ups
**Author:** Eli Holmes (NOAA)</br>

This notebook compares `point-collocation` to other approaches and shows how the package can speed up your collocation tasks.


## Load points

This is 15k+ of BGC-Argo points.

In [1]:
import pandas as pd
df = pd.read_parquet("https://raw.githubusercontent.com/fish-pace/fish-pace-datasets/main/datasets/chla_z/data/CHLA_argo_profiles.parquet")
df_points = df[["TIME", "LATITUDE", "LONGITUDE"]].rename(
    columns={
        "TIME": "time",
        "LATITUDE": "lat",
        "LONGITUDE": "lon"
    }
)
print(len(df_points))
df_points.head()

15833


,time,lat,lon
0,2024-03-01 21:23:16.002000128,54.6582,-19.2434
1,2024-03-11 20:45:53.002000128,54.9187,-18.9609
2,2024-03-21 21:21:39.002000128,55.2967,-18.8331
3,2024-03-31 21:31:53.002000128,55.7268,-18.8653
4,2024-03-07 18:01:17.002000128,17.6665,-46.0155


# Use point-collocation

## We create a plan for these points

15k points takes about 15 seconds to search EarthData catalog and develop a plan.

In [2]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_RRS",
        "granule_name": "*.DAY.*.4km.*",
    }
)

CPU times: user 970 ms, sys: 67.8 ms, total: 1.04 s
Wall time: 7.64 s


In [14]:
plan.summary(n=0)

Plan: 15833 points → 620 unique granule(s)
  Points with 0 matches : 416
  Points with >1 matches: 0
  Time buffer: 0 days 00:00:00


## Now matchup

This takes about 2 hours capping out at about 5Gb of RAM.

In [ ]:
%%time
res = pc.matchup(plan, geometry="grid", variables=["Rrs"], 
                 save_dir="_temp_data",
                batch_size=10)

granules 1-10 of 620 processed, 160 points matched


# Workflow with subfunctions

This processes files in batches similar to `pc.matchup()` with `save_dir` used.

* Create a function to get all matches for one PACE file `one_file_matches()`
* Create a function to run a batch of PACE files `run_batch()`
* Do a for loop over our batches and run `run_batch()` on each
* Gather results into one file

## `one_file_matches()`



In [9]:
## The function to process one file
from typing import Optional, Tuple
import xarray as xr
import pandas as pd
import numpy as np

def one_file_matches(
    f: "earthaccess.store.EarthAccessFile",
    df: pd.DataFrame,
    ds_lat_name: str = "lat",
    ds_lon_name: str = "lon",
    ds_time_name: str = "time",
    ds_vec_name: Optional[str] = "wavelength",
    ds_var_name: str = "Rrs",
    ds_vec_sel = None,
    df_lat_name: str = "lat",
    df_lon_name: str = "lon",
    df_time_name: str = "time",
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    """
    Match Argo point observations to a single PACE L2/L3 file and extract
    colocated satellite values and metadata.

    Parameters
    ----------
    f : file-like object
        An earthaccess/open file-like handle for a single PACE granule
        (as returned by `earthaccess.open`). This object is passed directly
        to `xr.open_dataset` to read the granule.
    df : pandas.DataFrame
        A DataFrame containing Argo observations. Must include columns for
        time, latitude, longitude, and the target Argo variable to be matched.

    ds_lat_name, ds_lon_name, ds_time_name : str, optional
        Names of the latitude, longitude, and time variables in the PACE
        dataset. Default: "lat", "lon", "time".

    ds_vec_name : str or None, optional
        Name of the spectral dimension in the PACE dataset (e.g. "wavelength").
        If not None, matched satellite spectra are returned with one column per
        wavelength. If None, only a single variable is extracted.

    ds_vec_sel : value or None, optional
        Value of the spectral dimension in the PACE dataset (e.g. "wavelength")
        to select. If None, matched satellite spectra are returned with one column per
        wavelength. If given, only a single variable is extracted for that value.

    ds_var_name : str, optional
        Name of the satellite variable to extract from the PACE dataset
        (e.g. "Rrs" or "chlor_a").

    df_lat_name, df_lon_name, df_time_name : str, optional
        Column names in `df` for latitude, longitude, and time.

    df_var_name : str, optional
        Column name in `df` for the Argo variable being matched.

    Returns
    -------
    df_record : pandas.DataFrame or None
        A subset of `df` containing only the Argo observations whose timestamps
        fall within the PACE file's temporal coverage window. Returns None if
        no observations fall within this window.

    pts : pandas.DataFrame or None
        A DataFrame containing colocated satellite values for each matched Argo
        observation, including:
            - matched PACE pixel coordinates
            - the Argo variable value
            - spectral satellite values (if `ds_vec_name` is provided)
            - PACE temporal metadata (`pace_t_start`, `pace_t_end`)
            - the PACE file name (`pace_file`)
        Returns None if no points were matched.

Examples
    --------
    >>> import earthaccess
    >>> import pandas as pd
    >>>
    >>> # Log in and search for PACE granules (simplified example)
    >>> earthaccess.login()
    >>> results = earthaccess.search(
    ...     short_name="PACE_OCI_L3M_DAY_IOP",
    ...     temporal=("2024-03-05", "2024-03-06"),
    ...     bounding_box=(-180, -90, 180, 90),
    ... )
    >>> files = earthaccess.open(results, pqdm_kwargs={"disable": True})
    >>>
    >>> # Load Argo matchup candidates
    >>> df_argo = pd.read_parquet("tutorial_data/chl_argo_points.parquet")
    >>>
    >>> # Match a single PACE file to the Argo DataFrame
    >>> df_record, pts = one_file_matches(
    ...     files[0],
    ...     df_argo,
    ...     ds_vec_name=None,            # e.g. non-spectral variable like "chlor_a"
    ...     ds_var_name="chlor_a",
    ...     df_var_name="argo_chl"
    ... )
    
    Notes
    -----
    - Time matching uses a half-open interval: `[t_start, t_end)`.
    - Spatial matching is performed using nearest-neighbor lookup on the PACE
      latitude/longitude grid.
    - This function does not load full PACE granules into memory; only metadata
      and the required pixel values are accessed.
    - Returned DataFrames are aligned row-by-row: each row in `pts` corresponds
      to the same row in `df_record`.
    """
    with xr.open_dataset(f, chunks={}, cache=False) as ds:

        # --- time window in ds ---
        t_start = pd.to_datetime(ds.attrs["time_coverage_start"], utc=True)
        t_end   = pd.to_datetime(ds.attrs["time_coverage_end"], utc=True)

        # filename / product name
        fname = ds.attrs.get("product_name", None)
        if fname is None:
            src = ds.encoding.get("source", "")
            fname = src.split("/")[-1] if "/" in src else src

        df_times = pd.to_datetime(df[df_time_name], utc=True)
        # Use 24 hours so as not to cut off data if the window is small
        t_end_24 = t_start + pd.Timedelta(hours=24)
        df_record = df[(df_times >= t_start) & (df_times < t_end_24)]

        if df_record.empty:
            return None, None

        # --- spatial index / nearest lat-lon ---
        lat_idx = ds.get_index(ds_lat_name)
        lon_idx = ds.get_index(ds_lon_name)
        lat_vals = ds[ds_lat_name].values
        lon_vals = ds[ds_lon_name].values

        df_lats = df_record[df_lat_name].to_numpy(dtype=float)
        df_lons = df_record[df_lon_name].to_numpy(dtype=float)

        lat_i = lat_idx.get_indexer(df_lats, method="nearest")
        lon_i = lon_idx.get_indexer(df_lons, method="nearest")

        def sample_few_points(ds, lats, lons, var_name=ds_var_name):
            ds_var = ds[var_name]  # e.g. (lat, lon, wavelength)
            spectra = [
                ds_var.sel(lat=i, lon=j, method="nearest").values
                for i, j in zip(lats, lons)
            ]
            return np.stack(spectra, axis=0)

        var_vals = sample_few_points(ds, df_lats, df_lons)

        n = len(df_record)

        # --- build dataframe ---
        data = {
            # PACE file metadata per row
            f"pace_{ds_var_name}_file":  np.full(n, fname),
            f"pace_{ds_var_name}_t_start": np.full(n, t_start),
            f"pace_{ds_var_name}_t_end":   np.full(n, t_end),
            f"pace_{ds_var_name}_lat": lat_vals[lat_i],
            f"pace_{ds_var_name}_lon": lon_vals[lon_i],
        }

        if ds_vec_name is not None:
            vec_vals = ds[ds_vec_name].values
            if ds_vec_sel is not None:
                m = vec_vals == ds_vec_sel
                vec_vals = vec_vals[m]
            for j, v in enumerate(vec_vals):
                label = int(v)
                col_name = f"pace_{ds_var_name}_{label}"
                data[col_name] = var_vals[:, j].astype(float)
        else:
            col_name = f"pace_{ds_var_name}"
            data[col_name] = var_vals[:].astype(float)

        pts = pd.DataFrame(data)
        return df_record, pts

## `run_batch()`

In [10]:
%%time
from datetime import datetime

def run_batch(
    results, df, 
    ds_vec_name="wavelength", ds_var_name="Rrs", ds_vec_sel=None,
    df_time_name="time", df_lat_name="lat", df_lon_name="lon"
):
    """
    Run a batch of PACE files (results) and return PACE variables for lat/lon/time rows in a
    dataframe.

    Parameters
    ----------
    results : earthaccess results object as returned by `earthaccess.search`). 
    
    df : A pandas DataFrame containing Argo observations. Must include columns for
        time, latitude, longitude.

    ds_vec_name : str or None, optional
        Name of the spectral dimension in the PACE dataset (e.g. "wavelength").
        If not None, matched satellite spectra are returned with one column per
        wavelength. If None, only a single variable is extracted.

    ds_vec_sel : value or None, optional
        Value of the spectral dimension in the PACE dataset (e.g. "wavelength")
        to select. If None, matched satellite spectra are returned with one column per
        wavelength. If given, only a single variable is extracted for that value.

    ds_var_name : str, optional
        Name of the variable to extract from the PACE dataset
        (e.g. "Rrs" or "chlor_a").

    df_lat_name, df_lon_name, df_time_name : str, optional
        Column names in `df` for latitude, longitude, and time.
    """

    # Make sure to refresh fileset to minimize the chance that the token expires before we are done
    # this for loop consumes about 6Gb of RAM
    fileset = earthaccess.open(results, pqdm_kwargs={"disable": True} );
    
    df_plus = []
    for i, f in enumerate(fileset):
        df_record, pts = one_file_matches(
            f, df, 
            ds_vec_name=ds_vec_name, ds_var_name=ds_var_name, ds_vec_sel=ds_vec_sel,
            df_time_name=df_time_name, df_lat_name=df_lat_name, df_lon_name=df_lon_name)
        if df_record is None or len(df_record) == 0: 
            print(f"Skipped day {i} no data in df")
            continue
        # error check
        if len(df_record) != len(pts):
            raise ValueError(f"Row mismatch: df_record={len(df_record)}, pts={len(pts)}")
    
        # left "concat": keep df_record rows, add pts columns by position
        df_record_plus = pd.concat(
            [ 
                df_record.reset_index(drop=True), 
                pts.reset_index(drop=True),
            ], axis=1,)
        df_plus.append(df_record_plus)
    if not len(df_plus) == 0:
        df_plus = pd.concat(df_plus, ignore_index=True)
    return df_plus

CPU times: user 6 μs, sys: 1e+03 ns, total: 7 μs
Wall time: 8.58 μs


## Do the earthaccess search

In [7]:
import earthaccess
auth = earthaccess.login()
# are we authenticated?
if not auth.authenticated:
    # ask for credentials and persist them in a .netrc file
    auth.login(strategy="interactive", persist=True)
import xarray as xr
df = df_points
df['time'] = df['date']
results = earthaccess.search_data(
    short_name = "PACE_OCI_L3M_RRS",
    temporal = (df.time.min(), df.time.max()),
    granule_name="*.DAY.*.4km.*"
)
print(f"{len(results)} days of PACE data")

174 days of PACE data


## Run through all the files by batches

This takes about 4 hours for Rrs.

In [ ]:
%%time
from pathlib import Path
from datetime import datetime
import pandas as pd

results_dir = Path("_temp_data")
results_dir.mkdir(exist_ok=True)

all_batches = []
batch_size = 10
var = "CHLA"
results = results
df = df_points
ds_var = "Rrs"

for batch_idx, i in enumerate(range(0, len(results), batch_size), start=1):
    batch = results[i:i+batch_size]
    print(f"Batch {batch_idx}: {len(batch)} files: {datetime.now():%H:%M:%S}")

    # If this batch was already done, skip it
    batch_path = results_dir / f"{var}_matchups_{ds_var}_batch_{batch_idx:03d}.parquet"
    if batch_path.exists():
        print(f"  -> Skipping batch {batch_idx}, found {batch_path}")
        continue

    df_batch = run_batch(batch, df)  # returns a DataFrame

    # Save this batch immediately
    df_batch.to_parquet(batch_path, index=False)
    print(f"  -> Saved {len(df_batch)} rows to {batch_path}")


In [5]:
# When everything is done, you can merge all batches:
from pathlib import Path
import pandas as pd

results_dir = Path("_temp_data/matchups")
var = "CHLA"
ds_var = "Rrs"

batch_files = sorted(results_dir.glob(f"{var}_matchups_{ds_var}_batch_*.parquet"))
dfs = [pd.read_parquet(f) for f in batch_files]
df_all = pd.concat(dfs, ignore_index=True)
df_all.to_parquet(f"{results_dir}/{var}_argo_{ds_var}_all.parquet", index=False)


In [6]:
print(f"Total rows: {len(df_all)}")

Total rows: 18119


# Another smaller comparison

## This code takes ca 13 minutes

It maxes out at 1.32Gb.

In [1]:
import pandas as pd
# Load ship data
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/2025-tutorials/main/"
    "Supporting_files/SEAMAP_summary_2024_present.csv"
)
track_df = pd.read_csv(url)
# Load the dates and find all unique instances
track_df["Loc_date"] = pd.to_datetime(track_df["Loc_date"])
unique_dates = track_df["Loc_date"].sort_values().unique()
print(f"Number of unique dates to extract: {len(unique_dates)}")

Number of unique dates to extract: 61


In [2]:
%%time
import earthaccess
import xarray as xr
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

# Login
auth = earthaccess.login()
short_name = "PACE_OCI_L3M_RRS"
granule_name = "*.DAY.*.4km.*"
# Want 8D composites instead?
#granule_name = "*.8D.*.4km.*"
results_list = []

# Loop through unique dates
for date in tqdm(unique_dates, desc="Processing unique dates"):
    start = end = date.strftime("%Y-%m-%d")
    # Search & open files for exact date
    results = earthaccess.search_data(short_name=short_name, temporal=(start, end), granule_name=granule_name)
    fileset = earthaccess.open(results, pqdm_kwargs={"disable": True})

    for ea_file in fileset:  # iterate EarthAccessFile objects
        with xr.open_dataset(ea_file, engine="h5netcdf") as ds:
            points_today = track_df[track_df["Loc_date"] == date]
            
            for idx, row in points_today.iterrows():
                lat_center, lon_center = row["Latitude"], row["Longitude"]
                rrs = ds["Rrs"].sel(lat=lat_center, lon=lon_center, method="nearest")
                result_row = {'Original_Index': idx, **row.to_dict()}
                result_row.update({f"Rrs_{int(wl)}": val for wl, val in rrs.to_series().items()})
                results_list.append(result_row)

# Combine results & save
if results_list:
    final_df = pd.DataFrame(results_list).set_index("Original_Index").sort_index()
else:
    print("No data extracted. Check date ranges and data availability.")


Processing unique dates: 100%|██████████| 61/61 [13:48<00:00, 13.58s/it]

CPU times: user 24.8 s, sys: 2.89 s, total: 27.7 s
Wall time: 13min 50s


## Compare to point-collocation

2 minutes, 1.4 Gb

In [1]:
import pandas as pd
import pandas as pd
# Load ship data
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/2025-tutorials/main/"
    "Supporting_files/SEAMAP_summary_2024_present.csv"
)
track_df = pd.read_csv(url)
track_df["Loc_date"] = pd.to_datetime(track_df["Loc_date"])

df_points = track_df[["Loc_date", "Latitude", "Longitude"]].rename(
    columns={
        "Loc_date": "time",
        "Latitude": "lat",
        "Longitude": "lon"
    }
)
print(len(df_points))
df_points.head()

595


,time,lat,lon
0,2024-06-13,27.3835,-82.7375
1,2024-06-14,27.1190,-82.7125
2,2024-06-14,26.9435,-82.8170
3,2024-06-14,26.6875,-82.8065
4,2024-06-14,26.6675,-82.6455


In [2]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_RRS",
        "granule_name": "*.DAY.*.4km.*",
    }
)

CPU times: user 631 ms, sys: 69.5 ms, total: 701 ms
Wall time: 2.07 s


In [3]:
%%time
res = pc.matchup(plan, geometry="grid", variables=["Rrs"], 
                 save_dir="_temp_data",
                batch_size=10)

granules 1-10 of 61 processed, 141 points matched, 00:00:28
granules 11-20 of 61 processed, 116 points matched, 00:00:49
granules 21-30 of 61 processed, 61 points matched, 00:01:03
granules 31-40 of 61 processed, 59 points matched, 00:01:19
granules 41-50 of 61 processed, 133 points matched, 00:01:45
granules 51-60 of 61 processed, 83 points matched, 00:02:02
granules 61-61 of 61 processed, 2 points matched, 00:02:03
CPU times: user 1min 8s, sys: 5.31 s, total: 1min 14s
Wall time: 2min 3s
